# pdf_vlm — Document QA experiment (Colab)

## What this experiment measures

For Hyundai WIA report packs at **5 / 10 / 20 / 50 / 100 pages**:

| Axis | Variants |
|---|---|
| OCR | PP-StructureV3 (**tables ON** on Colab) + PDF text enrich |
| RAG generation | **text-only** vs **multimodal** (page images) |
| Retrieval | **page-level** vs **hierarchical** |
| Metric | ANLS / EM / F1 + recall@k (needs Gemma GGUF for real answers) |

**Runtime:** GPU (T4+) recommended.

> Do **not** clone into `/content/pdf_vlm` — that folder name shadows the Python package.

## 0. Clone + install (package + OCR + llama.cpp)

In [1]:
import sys, shutil
from pathlib import Path

REPO_URL = "https://github.com/mAn-He/pdf_vlm.git"
ROOT = Path("/content/pdf_vlm_repo")

# Remove shadowed clone path if present
shadow = Path("/content/pdf_vlm")
if shadow.exists() and shadow.resolve() != ROOT.resolve():
    shutil.rmtree(shadow, ignore_errors=True)

if not (ROOT / "pyproject.toml").exists():
    !git clone --depth 1 {REPO_URL} {ROOT}
else:
    print("Repo present:", ROOT)

%cd {ROOT}
for k in list(sys.modules):
    if k == "pdf_vlm" or k.startswith("pdf_vlm."):
        del sys.modules[k]
sys.path.insert(0, str(ROOT / "src"))
print("cwd:", Path.cwd())

Cloning into '/content/pdf_vlm_repo'...
remote: Enumerating objects: 216, done.
remote: Counting objects: 100% (216/216), done.
remote: Compressing objects: 100% (185/185), done.
remote: Total 216 (delta 33), reused 188 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (216/216), 1.21 MiB | 13.30 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/content/pdf_vlm_repo
cwd: /content/pdf_vlm_repo


In [2]:
import subprocess, sys
from pathlib import Path

ROOT = Path("/content/pdf_vlm_repo").resolve()
assert (ROOT / "src/pdf_vlm/utils/io.py").exists()

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"])
# index + OCR (tables) + viz
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[index,ocr,viz]"])

def try_install_llama():
    for url in [
        "https://abetlen.github.io/llama-cpp-python/whl/cu124",
        "https://abetlen.github.io/llama-cpp-python/whl/cu122",
        None,
    ]:
        cmd = [sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python"]
        if url:
            cmd += ["--extra-index-url", url]
        print("Trying llama-cpp:", url or "default")
        if subprocess.run(cmd).returncode == 0:
            return True
    return False

print("llama-cpp:", try_install_llama())

for k in list(sys.modules):
    if k == "pdf_vlm" or k.startswith("pdf_vlm."):
        del sys.modules[k]
sys.path.insert(0, str(ROOT / "src"))

import pdf_vlm
from pdf_vlm.utils.io import project_root
from pdf_vlm.ocr.paddle_structure import paddle_available
print("pdf_vlm:", pdf_vlm.__file__)
print("paddle:", paddle_available())

import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

Trying llama-cpp: https://abetlen.github.io/llama-cpp-python/whl/cu124
llama-cpp: True
pdf_vlm: /content/pdf_vlm_repo/src/pdf_vlm/__init__.py
paddle: {'paddleocr': True, 'PPStructureV3': True, 'PPStructure': False, 'PaddleOCR': True}
cuda: True Tesla T4


## 1. Download Gemma GGUF (required for real QA answers)

1. Accept: https://huggingface.co/google/gemma-3-4b-it-qat-q4_0-gguf  
2. Colab secret `HF_TOKEN` or paste token  
3. Set `DOWNLOAD_GGUF = True` below

If GGUF is missing, the harness can still measure **retrieval**, but **ANLS/QA quality will be empty** (dry-run).

In [3]:
from pathlib import Path
import os

DOWNLOAD_GGUF = True  # set False only if you intentionally skip generation

gguf = Path("models/gemma-3-4b-it-q4_0.gguf")
mmproj = Path("models/mmproj-model-f16-4B.gguf")

if DOWNLOAD_GGUF and not (gguf.exists() and mmproj.exists()):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        from getpass import getpass
        os.environ["HF_TOKEN"] = getpass("HF token: ")
    !huggingface-cli login --token "$HF_TOKEN" --add-to-git-credential
    !{sys.executable} scripts/download_models.py --with-mmproj

print("gguf:", gguf.exists(), gguf)
print("mmproj:", mmproj.exists(), mmproj)
HAS_GGUF = gguf.exists() and mmproj.exists()
print("HAS_GGUF:", HAS_GGUF)

HF token: ··········

Hint: A new version of huggingface_hub (1.24.0) is available! You are using version 1.23.0.
To update, run: hf update
Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help

[01:16:35] INFO pdf_vlm: Downloading google/gemma-3-4b-it-qat-q4_0-gguf / gemma-3-4b-it-q4_0.gguf

gemma-3-4b-it-q4_0.gguf: downloading bytes:   6% 181M/3.16G [00:01<00:11, 263MB/s, 13.0MB/s  ]
gemma-3-4b-it-q4_0.gguf: downloading bytes:  42% 1.32G/3.16G [00:04<00:04, 453MB/s,  105MB/s  ]
gemma-3-4b-it-q4_0.gguf: downloading bytes:  59% 1.86G/3.16G [00:05<00:02, 557MB/s,  142MB/s  ]
gemma-3-4b-it-q4_0.gguf: downloading bytes:  91% 2.88G/3.16G [00:06<00:00, 871MB/s,  208MB/s  ]
gemma-3-4b-it-q4_0.gguf: downloading bytes:  95% 3.00G/3.16G [00:06<00:00, 760MB/s,  220MB/s  ]


## 2. Build length packs 5/10/20/50/100 (optional if already in repo)

Repo already ships truncated PDFs under `data/custom/{5,10,20,50,100}/`.  
If you uploaded full `QA_report_HW.pdf` to the repo root, you can rebuild packs here.

In [4]:
from pathlib import Path
import sys

BUCKETS = "5,10,20,50,100"
src = Path("QA_report_HW.pdf")

if src.exists():
    !{sys.executable} scripts/prepare_hw_report_dataset.py --pdf {src} --buckets {BUCKETS}
else:
    print("No QA_report_HW.pdf in repo root — using existing data/custom packs.")

for b in [5, 10, 20, 50, 100]:
    d = Path(f"data/custom/{b}")
    pdfs = list(d.glob("*.pdf")) if d.exists() else []
    print(f"bucket={b}: pdfs={[p.name for p in pdfs]} q={(d/'questions.json').exists()}")

No QA_report_HW.pdf in repo root — using existing data/custom packs.
bucket=5: pdfs=['acme_demo_5pages.pdf', 'hyundai_wia_qa_report_5p.pdf'] q=True
bucket=10: pdfs=['hyundai_wia_qa_report_10p.pdf'] q=True
bucket=20: pdfs=['hyundai_wia_qa_report_20p.pdf'] q=True
bucket=50: pdfs=['hyundai_wia_qa_report_50p.pdf'] q=True
bucket=100: pdfs=['hyundai_wia_qa_report_100p.pdf'] q=True


## 3. OCR (PP-StructureV3, tables ON) + build retrieval indexes

Uses `configs/ocr/pp_structure_v3_colab.yaml` (`use_table_recognition: true`).  
This is the step that actually pulls OCR/table text used by RAG.

In [5]:
import sys

BUCKETS = "5,10,20,50,100"
# Start smaller if Colab RAM is tight: BUCKETS = "5,10,20"

!{sys.executable} scripts/colab_prepare_custom.py \
  --buckets {BUCKETS} \
  --no-stub \
  --enrich-pdf-text \
  --hash-embedder \
  --ocr-config ocr/pp_structure_v3_colab.yaml \
  --force

from pdf_vlm.utils.io import load_json, resolve_path
prep = load_json(resolve_path("data/custom/colab_prepared.json"))
print(prep)
assert prep.get("items"), "No docs prepared — check data/custom manifests/PDFs"

[01:17:14] INFO pdf_vlm: paddle_available={'paddleocr': True, 'PPStructureV3': True, 'PPStructure': False, 'PaddleOCR': True} table=True
[01:17:15] INFO pdf_vlm.pdf.render: Rendered 5 pages from hyundai_wia_qa_report_5p.pdf
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
[01:17:16] INFO pdf_vlm.ocr.paddle: Paddle subprocess structurev3 page 1/5
[2026-07-26 01:17:16,191] [    INFO] paddle_structure.py:472 - Paddle subprocess structurev3 page 1/5
[01:17:38] INFO pdf_vlm.ocr.paddle: Paddle subprocess structurev3 page 2/5
[2026-07-26 01:17:38,345] [    INFO] paddle_structure.py:472 - Paddle subprocess structurev3 page 2/5
[01:17:56] INFO pdf_vlm.ocr.paddle: Paddle subprocess structurev3 page 3/5
[2026-07-26 01:17:56,036

## 4. Run eval matrix (RAG variants × page lengths)

Cells: **text/multimodal × page/hierarchical × custom_{5,10,20,50,100}**  
If `HAS_GGUF` is False → forced dry-run (retrieval only, ANLS≈0).

In [9]:
# %cd /content/pdf_vlm_repo
# !git pull origin main

/content/pdf_vlm_repo
From https://github.com/mAn-He/pdf_vlm
 * branch            main       -> FETCH_HEAD
Already up to date.


In [11]:
# for ds in ["custom_5", "custom_10"]:
#     run("text", ds)
#     run("multimodal", ds)

NameError: name 'run' is not defined

In [6]:
import subprocess, sys
from pathlib import Path

gguf = Path("models/gemma-3-4b-it-q4_0.gguf")
mmproj = Path("models/mmproj-model-f16-4B.gguf")
HAS_GGUF = gguf.exists() and mmproj.exists()

cmd = [
    sys.executable, "scripts/run_eval_harness.py",
    "--config", "configs/experiments/eval_hw_wia_colab.yaml",
    "--datasets", "custom_5,custom_10,custom_20,custom_50,custom_100",
    "--pipelines", "text,multimodal",
    "--retrievals", "page,hierarchical",
    "--top-k", "3",
    "--device", "cuda",
]
if not HAS_GGUF:
    cmd.append("--dry-run")
    print("WARNING: no GGUF → dry-run (retrieval only). Re-run section 1.")
else:
    print("GGUF found → full QA generation")

print(" ".join(cmd))
subprocess.check_call(cmd)

GGUF found → full QA generation
/usr/bin/python3 scripts/run_eval_harness.py --config configs/experiments/eval_hw_wia_colab.yaml --datasets custom_5,custom_10,custom_20,custom_50,custom_100 --pipelines text,multimodal --retrievals page,hierarchical --top-k 3 --device cuda


0

In [7]:
from pathlib import Path
import json

runs = sorted(Path("results/runs").glob("eval_*"), key=lambda p: p.stat().st_mtime, reverse=True)
print("latest:", runs[0] if runs else None)
if runs:
    for name in ["summary.json", "report.md", "aggregates.json"]:
        p = runs[0] / name
        if p.exists():
            print("====", name, "====")
            txt = p.read_text(encoding="utf-8")
            print(txt[:5000])
            break

latest: results/runs/eval_eval_hw_wia_colab_20260726_021722
==== summary.json ====
{
  "run_id": "eval_eval_hw_wia_colab_20260726_021722",
  "n_rows": 0,
  "n_skipped": 20,
  "paths": {
    "predictions_csv": "/content/pdf_vlm_repo/results/runs/eval_eval_hw_wia_colab_20260726_021722/predictions.csv",
    "predictions_json": "/content/pdf_vlm_repo/results/runs/eval_eval_hw_wia_colab_20260726_021722/predictions.json",
    "aggregates_json": "/content/pdf_vlm_repo/results/runs/eval_eval_hw_wia_colab_20260726_021722/aggregates.json",
    "table_overall": "/content/pdf_vlm_repo/results/runs/eval_eval_hw_wia_colab_20260726_021722/tables/overall.csv",
    "table_by_pipeline": "/content/pdf_vlm_repo/results/runs/eval_eval_hw_wia_colab_20260726_021722/tables/by_pipeline.csv",
    "table_by_retrieval": "/content/pdf_vlm_repo/results/runs/eval_eval_hw_wia_colab_20260726_021722/tables/by_retrieval.csv",
    "table_by_cell": "/content/pdf_vlm_repo/results/runs/eval_eval_hw_wia_colab_20260726_021722

## 5. Inference practicality bench (needs GGUF)

This cell intentionally skips if weights are missing — it is **not** the QA harness.

In [8]:
import sys
from pathlib import Path

gguf = Path("models/gemma-3-4b-it-q4_0.gguf")
if gguf.exists():
    !{sys.executable} scripts/bench_gemma_inference.py --repeats 2
else:
    print("Skip bench: models/gemma-3-4b-it-q4_0.gguf missing.")
    print("Fix: set DOWNLOAD_GGUF=True in section 1 and re-run that cell.")

[02:17:26] INFO pdf_vlm.llm.gemma: Using MTMDChatHandler with mmproj=/content/pdf_vlm_repo/models/mmproj-model-f16-4B.gguf
[02:17:26] INFO pdf_vlm.llm.gemma: Loading Gemma 3 with vision (mmproj=mmproj-model-f16-4B.gguf)
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
add_text: <start_of_turn>user
Say OK<end_of_turn>
<start_of_turn>model

[02:17:28] INFO pdf_vlm.bench.inference: Model ready (first-call includes load) first_call_ms=2570.2 rss=2122.5
[02:17:28] INFO pdf_vlm.bench.inference: warmup 1/1
add_text: <start_of_turn>user
You are a document QA assistant. Use ONLY the evidence below.

=== EVIDENCE (top-1 of 5-page doc) ===
[Page 0 of 5] Acme Corp section 0. Founded in 1998. Product VisionX-4. Revenue context for retrieval unit 0. lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lo

In [13]:
import json, subprocess, sys, gc
from pathlib import Path

%cd /content/pdf_vlm_repo

id_map = {}
for d in Path("indices").iterdir():
    if d.is_dir() and (d / "page_text").exists():
        id_map[d.name.rsplit("_", 1)[0]] = d.name
print(id_map)

for n in [5, 10, 20, 50, 100]:
    key = f"hyundai_wia_qa_report_{n}p"
    new_id = id_map.get(key)
    if not new_id:
        print("NO INDEX", key); continue
    for fname in ["questions.json", "manifest.json"]:
        p = Path(f"data/custom/{n}/{fname}")
        if not p.exists(): continue
        data = json.loads(p.read_text(encoding="utf-8"))
        if isinstance(data, list):
            for r in data: r["doc_id"] = new_id
        else:
            for d in data.get("documents") or []: d["doc_id"] = new_id
        p.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    print("synced", n, "->", new_id)

try:
    import torch; torch.cuda.empty_cache()
except: pass

def run(pipelines, datasets):
    cmd = [
        sys.executable, "scripts/run_eval_harness.py",
        "--config", "configs/experiments/eval_hw_wia_colab.yaml",
        "--datasets", datasets,
        "--pipelines", pipelines,
        "--retrievals", "page,hierarchical",
        "--top-k", "3",
        "--device", "cuda",
    ]
    print(">>>", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print("exit:", r.returncode)
    print(r.stdout[-3000:])
    if r.returncode != 0: print(r.stderr[-3000:])
    gc.collect()
    try: torch.cuda.empty_cache()
    except: pass
    return r.returncode

for ds in ["custom_5", "custom_10", "custom_20", "custom_50", "custom_100"]:
    assert run("text", ds) == 0
    assert run("multimodal", ds) == 0

/content/pdf_vlm_repo
{'hyundai_wia_qa_report_5p': 'hyundai_wia_qa_report_5p_bf5a4d3901', 'hyundai_wia_qa_report_50p': 'hyundai_wia_qa_report_50p_d93f25fcd0', 'hyundai_wia_qa_report_10p': 'hyundai_wia_qa_report_10p_c15a763b2e', 'hyundai_wia_qa_report_20p': 'hyundai_wia_qa_report_20p_08b16bcce1', 'hyundai_wia_qa_report_100p': 'hyundai_wia_qa_report_100p_c93681d10f'}
synced 5 -> hyundai_wia_qa_report_5p_bf5a4d3901
synced 10 -> hyundai_wia_qa_report_10p_c15a763b2e
synced 20 -> hyundai_wia_qa_report_20p_08b16bcce1
synced 50 -> hyundai_wia_qa_report_50p_d93f25fcd0
synced 100 -> hyundai_wia_qa_report_100p_c93681d10f
>>> /usr/bin/python3 scripts/run_eval_harness.py --config configs/experiments/eval_hw_wia_colab.yaml --datasets custom_5 --pipelines text --retrievals page,hierarchical --top-k 3 --device cuda
exit: 0
s.or.kr\nPage 3'
  [drop] hyundai_wia_qa_report_5p_bf5a4d3901::page::1 score=0.2682 pages=[1] preview='[대표이사 등의확인]\n대표이사등의확인·서명\n확인서\n우리는 당사의 대표이사및 신고업무담당이사로서 이 보고서의 기재내용에 대해 상당\n한 

In [14]:
import shutil
from pathlib import Path
from datetime import datetime

ROOT = Path("/content/pdf_vlm_repo")
export = Path("/content/pdf_vlm_export")
if export.exists(): shutil.rmtree(export)
export.mkdir()

for rel in ["results/runs", "results/reports", "results/figures", "results/bench"]:
    src = ROOT / rel
    if src.exists():
        shutil.copytree(src, export / rel, dirs_exist_ok=True)
        print("copied", rel)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_base = f"/content/pdf_vlm_all_results_{stamp}"
shutil.make_archive(zip_base, "zip", export)
zip_file = Path(zip_base + ".zip")
print("ZIP:", zip_file, "MB=", round(zip_file.stat().st_size/1e6, 2))

from google.colab import files
files.download(str(zip_file))

copied results/runs
copied results/reports
copied results/figures
copied results/bench
ZIP: /content/pdf_vlm_all_results_20260726_042652.zip MB= 1.32


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>